# Cross-Embodiment Training: 人魚への転移実験

> 「形を覚えるな、意図を覚えろ。」

このノートブックは **Cross-Embodiment Training** の核心的な主張を、  
「二足歩行 + 魚の泳ぎ → 人魚への転移」という実験で検証します。

---

## このノートブックで何を証明するか

| 条件 | 事前知識 | 人魚データ量 |
|------|----------|-------------|
| **A（転移あり）** | 二足 + 魚で事前学習済み | 少量（N件） |
| **B（ベースライン）** | ゼロ（ランダム初期化） | 同じN件 |

**主張：** 同じデータ量でも、AはBより速く・精度高く人魚の動きを学習できる。  
これが「データ資産が蓄積される」という意味です。

---

## 章構成

```
Chapter 1: なぜCross-Embodiment Trainingが必要か
Chapter 2: 学習データの生成（二足 / 魚 / 人魚）
Chapter 3: モデルアーキテクチャの設計
Chapter 4: 事前学習（二足 + 魚）
Chapter 5: 人魚への転移実験（条件A vs B）
Chapter 6: 結果の解釈と限界
```

---
# Chapter 1: なぜCross-Embodiment Trainingが必要か

## 従来のアプローチの問題

通常のキャラクターAIは「一体専用モデル」を作ります。

```
二足キャラ専用モデル  → 二足しか動かせない
魚専用モデル         → 魚しか動かせない
人魚を追加したい     → ゼロから学習し直し（コスト大）
```

問題の本質は、**「身体の形」と「タスクの意図」が同じモデルに混在している**ことです。

## Cross-Embodiment Trainingの発想

異なる身体のデータを混合して学習させることで、  
モデルに「身体の形に依存しない意図の表現」を学ばせます。

```
二足のデータ ─┐
魚のデータ   ─┼─→ 共通Backbone → 「移動する」「追う」の意図を汎化
              └
                       ↓
            人魚追加 → ActionExpertだけ少量データで学習
```

## π₀（Physical Intelligence）との共通原理

Google DeepMindのRT-2やPhysical Intelligenceのπ₀も同じ原理を使っています。  
π₀が「見たことのない洗濯機」を操作できるのは、  
**「扉を開ける」「ボタンを押す」という意図を抽象化しているから**です。

本実験では、π₀の考え方をゲームキャラクターのモーション学習に応用します。

---
# Chapter 2: 学習データの生成

## なぜ「合成データ」を使うか

実際の動作データ（モーションキャプチャなど）は存在しないため、  
**生体力学の研究に基づいた数式**から時系列トラジェクトリを生成します。

重要なのは「データが物理的に正確かどうか」ではなく、  
**「条件A・Bに同じデータを与えたとき、転移が有効かどうか」** を測ることです。

## State / Action の定義

| 変数 | 次元 | 説明 |
|------|------|------|
| `state` | 10次元 | dist, dir_x, dir_z, sin(phase), cos(phase), speed, water_depth, height, task_id, obstacle |
| `spec` | 5次元 | Embodiment Token（身体スペック） |
| `action` | 8次元 | 各キャラクター共通の出力次元（関節角度） |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0a0a0f'
plt.rcParams['axes.facecolor']   = '#111118'
plt.rcParams['text.color']       = '#cccccc'
plt.rcParams['axes.labelcolor']  = '#cccccc'
plt.rcParams['xtick.color']      = '#888888'
plt.rcParams['ytick.color']      = '#888888'
plt.rcParams['axes.edgecolor']   = '#333344'
plt.rcParams['grid.color']       = '#1a1a2e'

STATE_DIM  = 10
SPEC_DIM   = 5
ACTION_DIM = 8
SEQ_LEN    = 60

# Embodiment Token: [move_speed, leg_count_norm, has_tail, water_ability, body_length_norm]
SPEC_BIPED   = np.array([0.6, 0.4, 0.0, 0.1, 0.3], dtype=np.float32)
SPEC_FISH    = np.array([0.5, 0.0, 1.0, 1.0, 0.7], dtype=np.float32)
SPEC_MERMAID = np.array([0.55, 0.2, 1.0, 0.7, 0.5], dtype=np.float32)

print('Embodiment Token:')
print(f'  二足   : {SPEC_BIPED}')
print(f'  魚     : {SPEC_FISH}')
print(f'  人魚   : {SPEC_MERMAID}')
print()
print('人魚のSpecは二足と魚の中間値 → Backboneに「2つの意図を混ぜてほしい」というヒント')

In [ ]:
def make_state(t, phase, task_id, water_depth=0.0, noise=0.02):
    dist     = 0.3 + 0.4 * abs(np.sin(t * 0.3))
    dir_x    = np.cos(t * 0.4)
    dir_z    = np.sin(t * 0.4)
    sin_p    = np.sin(phase)
    cos_p    = np.cos(phase)
    speed    = 0.6
    height   = 0.0
    obstacle = 0.0
    state = np.array(
        [dist, dir_x, dir_z, sin_p, cos_p,
         speed, water_depth, height, task_id, obstacle],
        dtype=np.float32
    )
    state += np.random.normal(0, noise, STATE_DIM).astype(np.float32)
    return state


def generate_biped_trajectory(n_traj=2000, task_id=1.0):
    """二足歩行トラジェクトリ生成。
    物理的根拠: 股関節は正弦波（左右逆位相）、膝は片側整流、腕は逆位相スイング。
    """
    states, actions, specs = [], [], []
    for _ in range(n_traj):
        t0    = np.random.uniform(0, 100)
        freq  = np.random.uniform(2.4, 3.2)
        amp   = np.random.uniform(0.35, 0.55)
        phase = np.random.uniform(0, 2 * np.pi)
        dt    = 0.016
        traj_s, traj_a = [], []
        for step in range(SEQ_LEN):
            t = t0 + step * dt
            phase += freq * dt
            state = make_state(t, phase, task_id, water_depth=0.0)
            lHip  =  np.sin(phase) * amp
            rHip  = -np.sin(phase) * amp
            lKnee =  max(0, -np.sin(phase)) * amp * 1.2
            rKnee =  max(0,  np.sin(phase)) * amp * 1.2
            lArm  = -np.sin(phase) * 0.28
            rArm  =  np.sin(phase) * 0.28
            torso =  np.sin(phase * 2) * 0.04
            bob   =  abs(np.sin(phase)) * 0.03
            action = np.array([lHip, rHip, lKnee, rKnee, lArm, rArm, torso, bob], dtype=np.float32)
            action += np.random.normal(0, 0.01, ACTION_DIM).astype(np.float32)
            traj_s.append(state)
            traj_a.append(action)
        states.append(traj_s)
        actions.append(traj_a)
        specs.append(SPEC_BIPED)
    return np.array(states), np.array(actions), np.array(specs)


def generate_fish_trajectory(n_traj=2000, task_id=1.0):
    """魚の泳ぎトラジェクトリ生成。
    物理的根拠: Body Caudal Propulsion (BCP) モデル。
    体幹に後方伝播する進行波、振幅は尾ひれに向かって二次関数的に増大。
    参考: Lighthill (1960), Videler & Hess (1984)
    """
    states, actions, specs = [], [], []
    for _ in range(n_traj):
        t0          = np.random.uniform(0, 100)
        freq        = np.random.uniform(1.8, 2.6)
        waveK       = np.random.uniform(0.7, 1.1)
        phase       = np.random.uniform(0, 2 * np.pi)
        dt          = 0.016
        water_depth = np.random.uniform(0.6, 1.0)
        traj_s, traj_a = [], []
        for step in range(SEQ_LEN):
            t = t0 + step * dt
            phase += freq * dt
            state = make_state(t, phase, task_id, water_depth=water_depth)
            amp_base, amp_tail = 0.06, 0.55
            def seg(s):
                a = amp_base + (amp_tail - amp_base) * s ** 2
                return np.sin(phase - waveK * s * np.pi) * a
            action = np.array([
                seg(0.0), seg(0.33), seg(0.66), seg(1.0),
                 np.sin(phase * 0.7) * 0.14,
                -np.sin(phase * 0.7) * 0.14,
                 np.sin(phase * 0.4) * 0.04,
                 np.sin(phase * 0.3) * 0.02,
            ], dtype=np.float32)
            action += np.random.normal(0, 0.01, ACTION_DIM).astype(np.float32)
            traj_s.append(state)
            traj_a.append(action)
        states.append(traj_s)
        actions.append(traj_a)
        specs.append(SPEC_FISH)
    return np.array(states), np.array(actions), np.array(specs)


def generate_mermaid_trajectory(n_traj=500, task_id=1.0):
    """人魚トラジェクトリ生成。
    上半身（二足系の腕・胴体）+ 下半身（魚系の尾ひれ）を水深でブレンド。
    これが「単純な二足でも魚でもない第三の動き」であり、
    ゼロから学習するより転移の方が有効なことを示す難しさの源泉。
    """
    states, actions, specs = [], [], []
    for _ in range(n_traj):
        t0          = np.random.uniform(0, 100)
        freq        = np.random.uniform(2.0, 2.8)
        waveK       = 0.85
        amp         = np.random.uniform(0.30, 0.45)
        phase       = np.random.uniform(0, 2 * np.pi)
        dt          = 0.016
        water_depth = np.random.uniform(0.3, 1.0)
        blend       = water_depth
        traj_s, traj_a = [], []
        for step in range(SEQ_LEN):
            t = t0 + step * dt
            phase += freq * dt
            state  = make_state(t, phase, task_id, water_depth=water_depth)
            arm_l  = -np.sin(phase) * 0.28 * (1 - blend * 0.6)
            arm_r  =  np.sin(phase) * 0.28 * (1 - blend * 0.6)
            torso  =  np.sin(phase * 2) * 0.04
            bob    =  abs(np.sin(phase)) * 0.02 * (1 - blend)
            amp_base, amp_tail = 0.06, 0.55
            def seg(s):
                a = amp_base + (amp_tail - amp_base) * s ** 2
                return np.sin(phase - waveK * s * np.pi) * a * blend
            tail_mid  = seg(0.5) * amp
            tail_tip  = seg(1.0) * amp
            hip_pitch = np.sin(phase) * amp * 0.5 * blend
            roll      = np.sin(phase * 0.5) * 0.03
            action = np.array([arm_l, arm_r, torso, bob,
                                tail_mid, tail_tip, hip_pitch, roll], dtype=np.float32)
            action += np.random.normal(0, 0.012, ACTION_DIM).astype(np.float32)
            traj_s.append(state)
            traj_a.append(action)
        states.append(traj_s)
        actions.append(traj_a)
        specs.append(SPEC_MERMAID)
    return np.array(states), np.array(actions), np.array(specs)


print('データ生成関数定義完了')

In [ ]:
# Japanese font setup for Colab
import subprocess
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-noto-cjk'], capture_output=True)
import matplotlib
matplotlib.font_manager._fmcache = {}
matplotlib.font_manager.fontManager.addfont('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
matplotlib.rcParams['font.family'] = 'Noto Sans CJK JP'
print('Japanese font installed')

print('Generating data...')
biped_s,   biped_a,   biped_spec   = generate_biped_trajectory(2000)
fish_s,    fish_a,    fish_spec    = generate_fish_trajectory(2000)
mermaid_s, mermaid_a, mermaid_spec = generate_mermaid_trajectory(500)

print(f'biped  : {biped_s.shape}')
print(f'fish   : {fish_s.shape}')
print(f'mermaid: {mermaid_s.shape}')

fig, axes = plt.subplots(3, 1, figsize=(12, 8))
fig.suptitle('Generated trajectories (first SEQ_LEN steps)', color='#dddddd', fontsize=13)

datasets = [
    (biped_a[0],   ['#4a9eff','#ff6b6b','#4eff9e','#ffaa44'],
     ['lHip','rHip','lKnee','rKnee'],         'Biped walking - hip/knee phase control'),
    (fish_a[0],    ['#1ae8d0','#0a8a7a','#5fffff','#00ddcc'],
     ['seg0(head)','seg1','seg2','seg3(tail)'],'Fish swimming - BCP model (head to tail wave)'),
    (mermaid_a[0], ['#c86bff','#8855dd','#ee99ff','#aa66ee'],
     ['arm_l','arm_r','tail_mid','tail_tip'],  'Mermaid - upper body (arms) + lower body (tail)'),
]
for ax, (data, colors, labels, title) in zip(axes, datasets):
    for i, (c, l) in enumerate(zip(colors, labels)):
        ax.plot(range(SEQ_LEN), data[:SEQ_LEN, i], color=c, linewidth=1.4, label=l)
    ax.set_title(title, color='#cccccc', fontsize=11)
    ax.legend(loc='upper right', fontsize=9, framealpha=0.3)
    ax.set_ylim(-0.7, 0.7)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

print()
print('Check: biped lHip/rHip should be opposite phase')
print('Check: fish seg0(head) amplitude small, seg3(tail) amplitude large')
print('Check: mermaid has both arm and tail signals')

---
# Chapter 3: モデルアーキテクチャの設計

```
┌──────────────────────────────────────────────────┐
│  EmbodimentEncoder                               │
│  Spec(5) → Linear(5→32) → ReLU → Linear(32→32)  │
│  「この身体は何ができるか」を潜在ベクトルに変換    │
└─────────────────────┬────────────────────────────┘
                      │
        [state(10) + emb(32)] = 42次元
                      │
┌─────────────────────▼────────────────────────────┐
│  SharedBackbone（全キャラ共通・転移の核心）        │
│  Linear(42→128) → ReLU → Linear(128→128) → ReLU │
│  → Linear(128→64)                                │
│  「何をしたいか（意図）」を身体によらず共有表現へ  │
└─────────────────────┬────────────────────────────┘
                      │ latent(64)
           ┌──────────┴──────────┐
           ▼                     ▼
┌──────────────────┐   ┌──────────────────┐
│  ActionExpert    │   │  ActionExpert    │
│  (biped / fish)  │   │  (mermaid)       │
│  64→32→8+tanh   │   │  64→32→8+tanh   │
│  ← 事前学習済み  │   │  ← 少量データで  │
│                  │   │    学習（転移）   │
└──────────────────┘   └──────────────────┘
```

**転移実験での違い:**
- **条件A**: 事前学習済みBackboneを凍結 → ActionExpertだけ学習
- **条件B**: 全パラメータをランダム初期化から学習

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')


class EmbodimentEncoder(nn.Module):
    """身体スペック → 潜在ベクトル。
    Specをそのまま使わず非線形変換することで
    Backboneが扱いやすい表現空間に射影する。
    """
    def __init__(self, spec_dim=5, emb_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(spec_dim, 32), nn.ReLU(),
            nn.Linear(32, emb_dim), nn.ReLU()
        )
    def forward(self, spec):
        return self.net(spec)


class SharedBackbone(nn.Module):
    """[state + emb] → latent(64)。
    全キャラクター共通。ここが転移の核心。
    """
    def __init__(self, in_dim=42, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(),
            nn.Linear(128, 128),   nn.ReLU(),
            nn.Linear(128, latent_dim), nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)


class ActionExpert(nn.Module):
    """latent(64) → action(8)。
    キャラクター固有の身体変換を担当。
    転移時はここだけ少量データで学習する。
    """
    def __init__(self, latent_dim=64, action_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(),
            nn.Linear(32, action_dim), nn.Tanh()
        )
    def forward(self, latent):
        return self.net(latent)


class CrossEmbodimentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder  = EmbodimentEncoder(SPEC_DIM, 32)
        self.backbone = SharedBackbone(STATE_DIM + 32, 64)
        self.expert   = ActionExpert(64, ACTION_DIM)

    def forward(self, state, spec):
        emb    = self.encoder(spec)
        inp    = torch.cat([state, emb], dim=-1)
        latent = self.backbone(inp)
        action = self.expert(latent)
        return action, latent


m = CrossEmbodimentModel()
total    = sum(p.numel() for p in m.parameters())
bb_p     = sum(p.numel() for p in m.backbone.parameters())
exp_p    = sum(p.numel() for p in m.expert.parameters())
print(f'総パラメータ数     : {total:,}')
print(f'Backboneパラメータ : {bb_p:,}  ← 事前学習で鍛える')
print(f'ActionExpert      : {exp_p:,}   ← 転移時だけ学習')

---
# Chapter 4: 事前学習（二足 + 魚）

二足と魚のデータを混合してSharedBackboneを学習します。

この段階でBackboneは：
- 「移動する」という意図の表現を獲得する
- 身体スペックに応じて適切な潜在表現に切り替える

**人魚のデータはまだ一切使いません。**

In [ ]:
def prepare_dataset(states, actions, specs):
    N, T, _ = states.shape
    s  = states.reshape(N * T, -1)
    a  = actions.reshape(N * T, -1)
    sp = np.repeat(specs, T, axis=0)
    return (
        torch.tensor(s,  dtype=torch.float32),
        torch.tensor(a,  dtype=torch.float32),
        torch.tensor(sp, dtype=torch.float32)
    )

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0
    for s, a, sp in loader:
        s, a, sp = s.to(device), a.to(device), sp.to(device)
        pred, _ = model(s, sp)
        loss = criterion(pred, a)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_loss(model, loader, criterion):
    model.eval()
    total = 0
    for s, a, sp in loader:
        s, a, sp = s.to(device), a.to(device), sp.to(device)
        pred, _ = model(s, sp)
        total += criterion(pred, a).item()
    return total / len(loader)

# ---- 事前学習データ作成 ----
pre_s  = np.concatenate([biped_s,    fish_s])
pre_a  = np.concatenate([biped_a,    fish_a])
pre_sp = np.concatenate([biped_spec, fish_spec])
idx    = np.random.permutation(len(pre_s))
pre_s, pre_a, pre_sp = pre_s[idx], pre_a[idx], pre_sp[idx]

split   = int(len(pre_s) * 0.9)
tr_s, tr_a, tr_sp = prepare_dataset(pre_s[:split], pre_a[:split], pre_sp[:split])
va_s, va_a, va_sp = prepare_dataset(pre_s[split:], pre_a[split:], pre_sp[split:])
train_loader = DataLoader(TensorDataset(tr_s, tr_a, tr_sp), batch_size=512, shuffle=True)
val_loader   = DataLoader(TensorDataset(va_s, va_a, va_sp), batch_size=512)

print(f'事前学習データ: train {len(tr_s):,} steps / val {len(va_s):,} steps')
print('人魚データは含まない')

# ---- 学習 ----
pretrained_model = CrossEmbodimentModel().to(device)
optimizer  = optim.Adam(pretrained_model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60)
criterion  = nn.MSELoss()

pretrain_losses = {'train': [], 'val': []}
PRETRAIN_EPOCHS = 60

print(f'\n事前学習開始（{PRETRAIN_EPOCHS} epochs）...')
for epoch in range(PRETRAIN_EPOCHS):
    tl = train_epoch(pretrained_model, train_loader, optimizer, criterion)
    vl = eval_loss(pretrained_model, val_loader, criterion)
    scheduler.step()
    pretrain_losses['train'].append(tl)
    pretrain_losses['val'].append(vl)
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}/{PRETRAIN_EPOCHS}  train={tl:.5f}  val={vl:.5f}')

print('事前学習完了')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pretrain_losses['train'], color='#4a9eff', lw=1.5, label='train')
ax.plot(pretrain_losses['val'],   color='#ff6b6b', lw=1.5, label='val', linestyle='--')
ax.set_title('事前学習損失曲線（二足 + 魚）', color='#cccccc')
ax.set_xlabel('epoch'); ax.set_ylabel('MSE loss')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

---
# Chapter 5: 人魚への転移実験

同じ人魚データをN件与えたとき：

- **条件A（転移）**: 事前学習済みBackboneを凍結し、ActionExpertだけ学習
- **条件B（ベースライン）**: 全パラメータをランダム初期化から学習

N = 10, 50, 100, 200 件で比較します。  
少ないデータほどAとBの差が大きいはずです。

In [ ]:
import copy

def finetune_expert_only(base_model, states, actions, specs, epochs=50, lr=1e-3):
    """条件A: Backboneを凍結してActionExpertだけ学習する。"""
    model = copy.deepcopy(base_model).to(device)
    for p in model.encoder.parameters():  p.requires_grad = False
    for p in model.backbone.parameters(): p.requires_grad = False
    opt  = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    crit = nn.MSELoss()
    s_t, a_t, sp_t = prepare_dataset(states, actions, specs)
    loader = DataLoader(TensorDataset(s_t, a_t, sp_t), batch_size=64, shuffle=True)
    history = []
    for _ in range(epochs):
        history.append(train_epoch(model, loader, opt, crit))
    return history

def train_from_scratch(states, actions, specs, epochs=50, lr=1e-3):
    """条件B: 全パラメータをランダム初期化から学習する。"""
    model = CrossEmbodimentModel().to(device)
    opt   = optim.Adam(model.parameters(), lr=lr)
    crit  = nn.MSELoss()
    s_t, a_t, sp_t = prepare_dataset(states, actions, specs)
    loader = DataLoader(TensorDataset(s_t, a_t, sp_t), batch_size=64, shuffle=True)
    history = []
    for _ in range(epochs):
        history.append(train_epoch(model, loader, opt, crit))
    return history

N_SAMPLES = [10, 50, 100, 200]
EPOCHS    = 50
N_RUNS    = 3
results   = {}

for n in N_SAMPLES:
    results[n] = {'A': [], 'B': []}
    print(f'\nN={n} トラジェクトリ...')
    for run in range(N_RUNS):
        idx  = np.random.choice(len(mermaid_s), n, replace=(n > len(mermaid_s)))
        m_s, m_a, m_sp = mermaid_s[idx], mermaid_a[idx], mermaid_spec[idx]
        hA = finetune_expert_only(pretrained_model, m_s, m_a, m_sp, EPOCHS)
        hB = train_from_scratch(m_s, m_a, m_sp, EPOCHS)
        results[n]['A'].append(hA)
        results[n]['B'].append(hB)
        print(f'  run{run+1}  A={hA[-1]:.5f}  B={hB[-1]:.5f}')

print('\n全実験完了')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('転移実験結果: 条件A（転移あり）vs 条件B（ベースライン）',
             color='#dddddd', fontsize=13)

for ax, n in zip(axes.flat, N_SAMPLES):
    hA  = np.array(results[n]['A'])
    hB  = np.array(results[n]['B'])
    xs  = range(1, EPOCHS + 1)
    muA, sdA = hA.mean(0), hA.std(0)
    muB, sdB = hB.mean(0), hB.std(0)

    ax.plot(xs, muA, color='#4a9eff', lw=2.0, label='A: 転移あり')
    ax.fill_between(xs, muA - sdA, muA + sdA, color='#4a9eff', alpha=0.15)
    ax.plot(xs, muB, color='#ff6b6b', lw=2.0, label='B: ベースライン', linestyle='--')
    ax.fill_between(xs, muB - sdB, muB + sdB, color='#ff6b6b', alpha=0.15)

    imp = (muB[-1] - muA[-1]) / muB[-1] * 100
    ax.set_title(f'N={n} トラジェクトリ  改善: {imp:.1f}%', color='#cccccc', fontsize=10)
    ax.set_xlabel('epoch'); ax.set_ylabel('MSE loss')
    ax.legend(fontsize=9, framealpha=0.3)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

# 改善率サマリ
fig2, ax2 = plt.subplots(figsize=(8, 4))
imps = [(np.array(results[n]['B']).mean(0)[-1] - np.array(results[n]['A']).mean(0)[-1])
        / np.array(results[n]['B']).mean(0)[-1] * 100 for n in N_SAMPLES]
bars = ax2.bar(range(len(N_SAMPLES)), imps,
               color=['#4a9eff','#5ab5ff','#6ac5ff','#7ad5ff'])
ax2.set_xticks(range(len(N_SAMPLES)))
ax2.set_xticklabels([f'N={n}' for n in N_SAMPLES])
ax2.set_title('データ量別: 転移による損失改善率', color='#cccccc')
ax2.set_ylabel('改善率 (%)')
ax2.grid(True, alpha=0.3, axis='y')
for bar, imp in zip(bars, imps):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{imp:.1f}%', ha='center', color='#aaddff', fontsize=10)
plt.tight_layout()
plt.show()

print('読み方: データが少ないほど改善率が高い = 少量データ時代に転移が特に有効')

---
# Chapter 6: 結果の解釈と限界

## 言えること

| 主張 | 根拠 |
|------|------|
| 「同じデータ量でも、転移ありの方が低い損失を達成できる」 | Chapter 5 学習曲線 |
| 「特にデータが少ない（N=10〜50）場面で効果が大きい」 | 改善率グラフ |
| 「身体の多様性がデータ資産になる」 | 二足+魚→人魚の実験構造 |

## 言えないこと

| 誤解されやすい主張 | 正確な言い方 |
|------|------|
| 「二足のデータだけで人魚が作れる」 | 人魚データは必要。ただし少量でよい |
| 「羽・水中など全く異なる身体にも転移できる」 | 同じ物理ドメイン内での転移 |
| 「合成データで証明された」 | 合成データでの有効性。実データは別途必要 |

## 次のステップ

```
1. 視覚入力化（Three.jsのcanvas → CNN encoder）
   数値Specを廃止 → 形状定義なしに転移できる方向へ

2. GNN Morphology Encoder
   身体スペックをグラフ構造で渡す
   → 「足が増えた」「尾ひれが2本」に対応可能

3. Flow Matching Head
   1ステップ出力 → Tステップのトラジェクトリ生成
   → π₀により近い実装へ
```

---
*このノートブックは「知性の解体新書」の技術実装シリーズの一部です。*

In [ ]:
import json, copy
from google.colab import files

def export_weights(model, filename):
    weights = {}
    for name, param in model.named_parameters():
        key = name.replace('.', '_')
        weights[key] = param.detach().cpu().numpy().flatten().tolist()
    with open(filename, 'w') as f:
        json.dump(weights, f)
    kb = len(json.dumps(weights)) / 1024
    print(f'保存: {filename}  ({kb:.1f} KB)')

# ============================================================
# 条件A: 転移あり — 事前学習Backbone + ActionExpert学習後
# ============================================================
# 転移実験のN=50のrun0モデルを取り出す
# （最もバランスのよいN=50を使用。好みでN=10等に変更可）
N_EXPORT = 50

# 条件Aのモデルを再学習して取得
idx  = np.random.choice(len(mermaid_s), N_EXPORT, replace=False)
m_s, m_a, m_sp = mermaid_s[idx], mermaid_a[idx], mermaid_spec[idx]

model_A = copy.deepcopy(pretrained_model).to(device)
for p in model_A.encoder.parameters():  p.requires_grad = False
for p in model_A.backbone.parameters(): p.requires_grad = False
opt_A = optim.Adam(filter(lambda p: p.requires_grad, model_A.parameters()), lr=1e-3)
crit  = nn.MSELoss()
s_t, a_t, sp_t = prepare_dataset(m_s, m_a, m_sp)
loader = DataLoader(TensorDataset(s_t, a_t, sp_t), batch_size=64, shuffle=True)
for epoch in range(80):
    model_A.train()
    for s, a, sp in loader:
        s, a, sp = s.to(device), a.to(device), sp.to(device)
        pred, _ = model_A(s, sp)
        loss = crit(pred, a)
        opt_A.zero_grad(); loss.backward(); opt_A.step()
    if (epoch+1) % 20 == 0:
        print(f'  A epoch {epoch+1}/80  loss={loss.item():.5f}')

# ============================================================
# 条件B: ベースライン — ランダム初期化からフル学習後
# ============================================================
model_B = CrossEmbodimentModel().to(device)
opt_B   = optim.Adam(model_B.parameters(), lr=1e-3)
for epoch in range(80):
    model_B.train()
    for s, a, sp in loader:
        s, a, sp = s.to(device), a.to(device), sp.to(device)
        pred, _ = model_B(s, sp)
        loss = crit(pred, a)
        opt_B.zero_grad(); loss.backward(); opt_B.step()
    if (epoch+1) % 20 == 0:
        print(f'  B epoch {epoch+1}/80  loss={loss.item():.5f}')

# ============================================================
# エクスポート
# ============================================================
export_weights(model_A, 'weights_A.json')
export_weights(model_B, 'weights_B.json')
files.download('weights_A.json')
files.download('weights_B.json')

print()
print('次のステップ:')
print('  デモHTMLを開き、weights_A.json と weights_B.json を')
print('  それぞれドロップして読み込む')